# Regularization: Ridge & Lasso

**Companion lesson:** https://ml-viz.vercel.app/courses/linear-regression/03-regularization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## Overfitting with correlated features

We build a dataset where only 3 of 20 features matter, then watch OLS, Ridge, and Lasso handle it.

In [ ]:
n, d = 60, 20
X = np.random.randn(n, d)
true_w = np.zeros(d); true_w[:3] = [3.0, -2.0, 1.5]
y = X @ true_w + 0.5 * np.random.randn(n)

# closed-form ridge: w = (X'X + lam*I)^-1 X'y   (lam=0 -> OLS)
def ridge(X, y, lam):
    return np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)

print("OLS weights (first 6):", np.round(ridge(X, y, 0)[:6], 2))
print("Ridge λ=10  (first 6):", np.round(ridge(X, y, 10)[:6], 2))

## Lasso via proximal gradient (ISTA)

The L1 penalty has no closed form, but coordinate-wise soft-thresholding solves it.

In [ ]:
def soft(z, t): return np.sign(z) * np.maximum(np.abs(z) - t, 0)

def lasso(X, y, lam, iters=500):
    w = np.zeros(X.shape[1]); L = np.linalg.norm(X, 2) ** 2
    for _ in range(iters):
        w = soft(w - X.T @ (X @ w - y) / L, lam / L)
    return w

w_lasso = lasso(X, y, lam=15)
print("Lasso non-zero weights:", np.flatnonzero(np.abs(w_lasso) > 1e-6))
print("values:", np.round(w_lasso[np.abs(w_lasso) > 1e-6], 2))

## Regularization paths

Watch every weight as λ sweeps — Ridge shrinks smoothly, Lasso snaps weights to exactly zero.

In [ ]:
lams = np.logspace(-2, 3, 40)
ridge_path = np.array([ridge(X, y, l) for l in lams])
lasso_path = np.array([lasso(X, y, l) for l in lams])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, path, name in [(axes[0], ridge_path, 'Ridge'), (axes[1], lasso_path, 'Lasso')]:
    for j in range(d):
        ax.plot(lams, path[:, j], color='#6366f1' if j < 3 else '#475569', lw=1.5 if j < 3 else 0.7)
    ax.set_xscale('log'); ax.set_xlabel('λ'); ax.set_title(f'{name} path')
axes[0].set_ylabel('weight value')
plt.tight_layout(); plt.show()
# purple = the 3 true features; gray = the 17 noise features

**Try it:** raise the noise level to 2.0, or make features correlated with `X[:, 3] = X[:, 0] + 0.01*np.random.randn(n)`. Watch what OLS does to those two weights vs. Ridge.